# MemoRAG

Memory-inspired RAG: compress the whole corpus into a global memory, then use it to draft retrieval clues.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

MemoRAG uses a light long-context model as *global memory* that sees the whole database, produces **clues** (draft answers / evidence hints) for the query, and uses those clues to retrieve. Here the memory is a hierarchical LLM summary.

In [ ]:
# build the memory: summarise groups of chunks, then summarise the summaries
group_summaries = [llm("Summarise the key facts:\n\n" + "\n".join(chunks[i:i + 5])) for i in range(0, len(chunks), 5)]
memory = llm("Merge these notes into one compact memory of the whole corpus (max 250 words):\n\n" + "\n\n".join(group_summaries))
print(memory)

Given the query, the memory drafts clues - concrete phrases that the evidence is likely to contain.

In [ ]:
def memorag(query, k=3):
    clues = llm_json(
        f"Memory of the corpus:\n{memory}\n\nWrite 3 short clue sentences that the evidence for this question would likely contain. "
        f"Reply as a JSON list of strings.\n\nQuestion: {query}"
    ) or []
    seen, hits = set(), []
    for q in [query] + [c for c in clues if isinstance(c, str)]:
        for h in store.search(q, k):
            if h["idx"] not in seen:
                seen.add(h["idx"]); hits.append(h)
    return clues, hits[:k + 2]

query = "Which techniques help reduce the cost of a RAG system?"
clues, hits = memorag(query)
print("clues:", clues)
print(answer(query, [h["text"] for h in hits]))